
# Project - Realized Volatility Timing (Heston SSM + UKF)

Objectif du notebook:
1. Estimer une volatilite realizee latente avec un **Unscented Kalman Filter (UKF)** applique a une dynamique de type Heston.
2. Construire le spread quotidien **implied - realized estimee**:
\[
 s_t = \sigma_{IV,t} - \hat\sigma_t
\]
3. Definir une execution **carry dynamique** en fonction de ce spread, puis comparer au carry statique.

> Notes pratiques
- Le calibrage rolling est couteux (optimisations successives). Le notebook est configure pour un compromis precision/temps.
- Les warnings `pyarrow` sur `sysctlbyname` peuvent apparaitre selon l'environnement; ils sont non bloquants.


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", 200)
np.set_printoptions(precision=4, suppress=True)



## 1) Parametres du projet

Tu peux ajuster ces hyperparametres pour accelerer ou raffiner l'estimation.


In [ ]:

# Donnees
DATA_PATH = "../data/spy_2020_2022.parquet"
START_DATE = "2020-01-02"
END_DATE = "2022-12-30"

# Construction IV ATM
TARGET_DTE = 30
MIN_DTE = 7
MAX_DTE = 90
ATM_BAND = 0.03
FALLBACK_NEAREST_M = 20

# UKF / Heston
TRADING_DAYS = 252
DT = 1.0 / TRADING_DAYS
R_MEAS = 1e-6

# Rolling calibration
ROLL_WINDOW = 252
RECALIB_FREQ = 21
MAXITER_OPT = 60

# Strategie
Z_WINDOW = 126
WEIGHT_SCALE = 2.0
TC_BPS = 2.0

print("Configuration chargee.")


## 2) Chargement et preparation des donnees options / spot

In [ ]:

option_cols = [
    "date",
    "expiration",
    "spot",
    "strike",
    "implied_volatility",
    "call_put",
    "volume",
]

df_opt = pd.read_parquet(DATA_PATH, columns=option_cols)
df_opt["date"] = pd.to_datetime(df_opt["date"])
df_opt["expiration"] = pd.to_datetime(df_opt["expiration"])

df_opt = df_opt[df_opt["date"].between(pd.Timestamp(START_DATE), pd.Timestamp(END_DATE))].copy()

df_opt["day_to_expiration"] = (df_opt["expiration"] - df_opt["date"]).dt.days
df_opt["moneyness"] = df_opt["strike"] / df_opt["spot"]

df_opt = df_opt[
    (df_opt["implied_volatility"] > 0)
    & (df_opt["day_to_expiration"] >= MIN_DTE)
    & (df_opt["day_to_expiration"] <= MAX_DTE)
].copy()

# Spot quotidien (median sur les lignes options du jour)
df_spot = (
    df_opt.groupby("date", as_index=False)["spot"]
    .median()
    .sort_values("date")
    .reset_index(drop=True)
)

df_spot["log_spot"] = np.log(df_spot["spot"])
df_spot["ret"] = df_spot["log_spot"].diff()

df_opt.shape, df_spot.shape


## 3) Extraction d'une IV ATM quotidienne (maturite cible ~30 jours)

In [ ]:

def compute_daily_atm_iv(
    df: pd.DataFrame,
    target_dte: int = 30,
    atm_band: float = 0.03,
    fallback_nearest_m: int = 20,
) -> pd.DataFrame:
    x = df.copy()
    x["abs_m"] = (x["moneyness"] - 1.0).abs()

    # Choix de l'expiration la plus proche de la maturite cible, date par date
    exp_map = (
        x.groupby(["date", "expiration"], as_index=False)["day_to_expiration"].first()
    )
    exp_map["abs_dte"] = (exp_map["day_to_expiration"] - target_dte).abs()
    idx = exp_map.groupby("date")["abs_dte"].idxmin()
    chosen = exp_map.loc[idx, ["date", "expiration"]]

    y = x.merge(chosen, on=["date", "expiration"], how="inner")

    out = []
    for d, g in y.groupby("date"):
        atm = g[g["abs_m"] <= atm_band]
        if atm.empty:
            atm = g.nsmallest(fallback_nearest_m, "abs_m")

        iv = atm["implied_volatility"].median()
        dte = atm["day_to_expiration"].median()
        out.append((d, iv, dte, len(atm)))

    df_iv = pd.DataFrame(out, columns=["date", "sigma_iv", "selected_dte", "n_options"])
    return df_iv.sort_values("date").reset_index(drop=True)


df_iv = compute_daily_atm_iv(
    df_opt,
    target_dte=TARGET_DTE,
    atm_band=ATM_BAND,
    fallback_nearest_m=FALLBACK_NEAREST_M,
)

df_iv.head()


## 4) UKF pour le modele d'etat de Heston

In [ ]:

EPS = 1e-10


def chol_psd(P: np.ndarray, jitter: float = 1e-9) -> np.ndarray:
    P = 0.5 * (P + P.T)
    eye = np.eye(P.shape[0])
    for i in range(8):
        try:
            return np.linalg.cholesky(P + (10**i) * jitter * eye)
        except np.linalg.LinAlgError:
            continue
    vals, vecs = np.linalg.eigh(P)
    vals = np.clip(vals, 1e-12, None)
    return vecs @ np.diag(np.sqrt(vals))


def sigma_points(x: np.ndarray, P: np.ndarray, alpha=0.15, beta=2.0, kappa=0.0):
    n = x.size
    lam = alpha**2 * (n + kappa) - n
    c = n + lam
    S = chol_psd(P)

    pts = np.zeros((2 * n + 1, n))
    pts[0] = x

    scale = np.sqrt(c)
    for i in range(n):
        col = scale * S[:, i]
        pts[i + 1] = x + col
        pts[n + i + 1] = x - col

    Wm = np.full(2 * n + 1, 1.0 / (2.0 * c))
    Wc = np.full(2 * n + 1, 1.0 / (2.0 * c))
    Wm[0] = lam / c
    Wc[0] = lam / c + (1 - alpha**2 + beta)

    return pts, Wm, Wc


def heston_transition(x: np.ndarray, params: dict, dt: float) -> np.ndarray:
    log_s, v = x
    v = max(v, 1e-8)

    mu = params["mu"]
    kappa = params["kappa"]
    theta = params["theta"]

    log_s_next = log_s + (mu - 0.5 * v) * dt
    v_next = v + kappa * (theta - v) * dt
    v_next = max(v_next, 1e-8)

    return np.array([log_s_next, v_next], dtype=float)


def process_covariance(x_pred: np.ndarray, params: dict, dt: float) -> np.ndarray:
    v = max(float(x_pred[1]), 1e-8)
    xi = params["xi"]
    rho = params["rho"]

    q11 = v * dt
    q22 = (xi**2) * v * dt
    q12 = rho * xi * v * dt

    Q = np.array([[q11, q12], [q12, q22]], dtype=float)
    return 0.5 * (Q + Q.T)


def ukf_step(y_obs, x, P, params, dt, R, alpha=0.15, beta=2.0, kappa=0.0):
    # Sigma points at t-1
    pts, Wm, Wc = sigma_points(x, P, alpha=alpha, beta=beta, kappa=kappa)

    # Prediction through transition
    pts_pred = np.array([heston_transition(p, params, dt) for p in pts])
    x_pred = np.sum(Wm[:, None] * pts_pred, axis=0)

    P_pred = np.zeros((2, 2), dtype=float)
    for i in range(pts_pred.shape[0]):
        d = (pts_pred[i] - x_pred).reshape(-1, 1)
        P_pred += Wc[i] * (d @ d.T)

    P_pred += process_covariance(x_pred, params, dt)

    # Measurement: y_t = log(S_t)
    Y = pts_pred[:, 0:1]
    y_pred = float(np.sum(Wm * Y[:, 0]))

    S = 0.0
    Pxy = np.zeros((2, 1), dtype=float)

    for i in range(Y.shape[0]):
        dy = float(Y[i, 0] - y_pred)
        dx = (pts_pred[i] - x_pred).reshape(2, 1)
        S += Wc[i] * dy * dy
        Pxy += Wc[i] * dx * dy

    S = S + R
    S = max(S, 1e-12)

    K = Pxy / S
    innovation = float(y_obs - y_pred)

    x_upd = x_pred + (K[:, 0] * innovation)
    x_upd[1] = max(x_upd[1], 1e-8)

    P_upd = P_pred - (K @ K.T) * S
    P_upd = 0.5 * (P_upd + P_upd.T)

    ll = -0.5 * (np.log(2.0 * np.pi * S) + (innovation**2) / S)
    return x_upd, P_upd, ll


def run_ukf(log_prices: np.ndarray, params: dict, dt=1/252, R=1e-6, x0=None, P0=None):
    n = len(log_prices)

    if x0 is None:
        init_v = np.nanvar(np.diff(log_prices[: min(40, n)])) * 252
        init_v = float(np.clip(init_v, 1e-4, 0.2))
        x = np.array([log_prices[0], init_v], dtype=float)
    else:
        x = x0.astype(float).copy()

    if P0 is None:
        P = np.diag([1e-4, 0.02]).astype(float)
    else:
        P = P0.astype(float).copy()

    states = np.zeros((n, 2), dtype=float)
    states[0] = x
    ll_sum = 0.0

    for t in range(1, n):
        x, P, ll = ukf_step(log_prices[t], x, P, params, dt, R)
        states[t] = x
        ll_sum += ll

    return states, ll_sum, x, P


## 5) MLE des parametres de Heston sur fenetre glissante

In [ ]:

def param_array_to_dict(a):
    return {
        "kappa": float(a[0]),
        "theta": float(a[1]),
        "xi": float(a[2]),
        "rho": float(a[3]),
        "mu": float(a[4]),
    }


def neg_loglik_on_window(param_array, log_prices_window, dt=1/252, R=1e-6):
    p = param_array_to_dict(param_array)

    # Penalite douce pour la condition de Feller: 2*kappa*theta >= xi^2
    feller_gap = 2.0 * p["kappa"] * p["theta"] - p["xi"] ** 2
    penalty = 0.0 if feller_gap >= 0 else 1e4 * (abs(feller_gap) ** 2)

    _, ll_sum, _, _ = run_ukf(log_prices_window, p, dt=dt, R=R)
    return -ll_sum + penalty


def calibrate_heston_params(log_prices_window, init_guess=None, maxiter=60):
    if init_guess is None:
        init_guess = np.array([4.0, 0.04, 0.60, -0.60, 0.00], dtype=float)

    bounds = [
        (1e-3, 20.0),   # kappa
        (1e-4, 1.0),    # theta
        (1e-3, 5.0),    # xi
        (-0.99, 0.99),  # rho
        (-1.0, 1.0),    # mu
    ]

    res = minimize(
        neg_loglik_on_window,
        x0=init_guess,
        args=(log_prices_window, DT, R_MEAS),
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": maxiter},
    )

    params = param_array_to_dict(res.x)
    return params, res


def rolling_ukf_heston(
    dates,
    log_prices,
    roll_window=252,
    recalib_freq=21,
    maxiter_opt=60,
):
    n = len(log_prices)
    v_hat = np.full(n, np.nan, dtype=float)

    param_cols = ["kappa", "theta", "xi", "rho", "mu"]
    p_store = {k: np.full(n, np.nan, dtype=float) for k in param_cols}

    init_guess = np.array([4.0, 0.04, 0.60, -0.60, 0.00], dtype=float)
    t = roll_window

    calibrations = []

    while t < n:
        w0 = t - roll_window
        w1 = t

        window_prices = log_prices[w0 : w1 + 1]
        params, res = calibrate_heston_params(window_prices, init_guess=init_guess, maxiter=maxiter_opt)
        init_guess = np.array([params[k] for k in param_cols], dtype=float)

        states_w, _, x_t, P_t = run_ukf(window_prices, params, dt=DT, R=R_MEAS)
        v_hat[t] = states_w[-1, 1]

        for k in param_cols:
            p_store[k][t] = params[k]

        calibrations.append({
            "date": dates[t],
            "success": bool(res.success),
            "fun": float(res.fun),
            **params,
        })

        next_t = min(t + recalib_freq, n - 1)
        for j in range(t + 1, next_t + 1):
            x_t, P_t, _ = ukf_step(log_prices[j], x_t, P_t, params, DT, R_MEAS)
            v_hat[j] = x_t[1]
            for k in param_cols:
                p_store[k][j] = params[k]

        t = next_t + 1

    out = pd.DataFrame({"date": dates, "v_hat": v_hat})
    for k in param_cols:
        out[k] = p_store[k]

    calib_df = pd.DataFrame(calibrations)
    return out, calib_df


## 6) Execution du rolling UKF + construction de la sigma realisee estimee

In [ ]:

df_model = df_spot[["date", "log_spot", "ret"]].dropna().reset_index(drop=True)

ukf_out, calib_log = rolling_ukf_heston(
    dates=df_model["date"].to_numpy(),
    log_prices=df_model["log_spot"].to_numpy(),
    roll_window=ROLL_WINDOW,
    recalib_freq=RECALIB_FREQ,
    maxiter_opt=MAXITER_OPT,
)

ukf_out["sigma_hat"] = np.sqrt(np.clip(ukf_out["v_hat"], 1e-8, None)) * np.sqrt(TRADING_DAYS)

print("Nombre de calibrations:", len(calib_log))
calib_log.tail()


In [ ]:

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax[0].plot(df_model["date"], df_model["spot"] if "spot" in df_model.columns else np.exp(df_model["log_spot"]), lw=1.2)
ax[0].set_title("SPY Spot")
ax[0].grid(True, alpha=0.3)

ax[1].plot(ukf_out["date"], ukf_out["sigma_hat"], lw=1.2, label="sigma_hat (UKF)")
ax[1].set_title("Volatilite annualisee estimee (UKF)")
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()


## 7) Spread implied - realized estimee

In [ ]:

df_signal = (
    df_iv[["date", "sigma_iv"]]
    .merge(ukf_out[["date", "sigma_hat"]], on="date", how="inner")
    .merge(df_model[["date", "ret"]], on="date", how="left")
    .sort_values("date")
    .reset_index(drop=True)
)

df_signal["spread"] = df_signal["sigma_iv"] - df_signal["sigma_hat"]

# Realized vol forward (proxy ex-post pour eval carry)
FWD_WINDOW = 5

df_signal["rv_fwd"] = (
    df_signal["ret"].rolling(FWD_WINDOW).std().shift(-FWD_WINDOW) * np.sqrt(TRADING_DAYS)
)

df_signal[["date", "sigma_iv", "sigma_hat", "spread", "rv_fwd"]].dropna().head()


In [ ]:

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax[0].plot(df_signal["date"], df_signal["sigma_iv"], label="sigma_IV", lw=1.2)
ax[0].plot(df_signal["date"], df_signal["sigma_hat"], label="sigma_hat (UKF)", lw=1.2)
ax[0].set_title("Implied vs Realized estimee")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(df_signal["date"], df_signal["spread"], lw=1.1, color="tab:purple")
ax[1].axhline(0.0, color="black", ls="--", lw=1)
ax[1].set_title("Spread s_t = sigma_IV - sigma_hat")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8) Strategie carry dynamique basee sur le spread

In [ ]:

def clip(x, lo, hi):
    return np.minimum(np.maximum(x, lo), hi)


df_bt = df_signal.copy()

# Z-score du spread pour calibrer l'intensite de position
roll_mean = df_bt["spread"].rolling(Z_WINDOW).mean()
roll_std = df_bt["spread"].rolling(Z_WINDOW).std()
df_bt["z_spread"] = (df_bt["spread"] - roll_mean) / roll_std

# Weight dynamique dans [-1, 1]
df_bt["w_dyn"] = clip(df_bt["z_spread"] / WEIGHT_SCALE, -1.0, 1.0)

# Base carry proxy (plus IV > RV future => plus favorable au short vol)
df_bt["carry_proxy"] = (df_bt["sigma_iv"] - df_bt["rv_fwd"]) / np.sqrt(TRADING_DAYS)

# Pas de look-ahead: on utilise le poids de la veille
w_lag = df_bt["w_dyn"].shift(1)

# Cout de turnover
turnover = (w_lag - w_lag.shift(1)).abs().fillna(0.0)
tc = (TC_BPS * 1e-4) * turnover

df_bt["ret_dyn"] = w_lag * df_bt["carry_proxy"] - tc
df_bt["ret_static"] = 1.0 * df_bt["carry_proxy"]

for c in ["ret_dyn", "ret_static"]:
    df_bt[f"nav_{c}"] = (1.0 + df_bt[c].fillna(0.0)).cumprod()

df_bt[["date", "ret_dyn", "ret_static", "nav_ret_dyn", "nav_ret_static"]].tail()


In [ ]:

def performance_stats(ret: pd.Series, name: str = "strategy") -> pd.Series:
    r = ret.dropna()
    ann_ret = r.mean() * TRADING_DAYS
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan

    cum = (1.0 + r).cumprod()
    dd = cum / cum.cummax() - 1.0
    max_dd = dd.min()

    hit = (r > 0).mean()

    return pd.Series(
        {
            "name": name,
            "ann_return": ann_ret,
            "ann_vol": ann_vol,
            "sharpe": sharpe,
            "max_drawdown": max_dd,
            "hit_ratio": hit,
            "n_obs": len(r),
        }
    )

stats = pd.concat(
    [
        performance_stats(df_bt["ret_dyn"], "carry_dynamic"),
        performance_stats(df_bt["ret_static"], "carry_static"),
    ],
    axis=1,
).T

stats


In [ ]:

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax[0].plot(df_bt["date"], df_bt["nav_ret_dyn"], label="Carry dynamique", lw=1.4)
ax[0].plot(df_bt["date"], df_bt["nav_ret_static"], label="Carry statique", lw=1.2, alpha=0.8)
ax[0].set_title("NAV comparee")
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(df_bt["date"], df_bt["w_dyn"], label="Poids dynamique", lw=1.0)
ax[1].axhline(0.0, color="black", ls="--", lw=1)
ax[1].set_title("Allocation dynamique w_t")
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()



## 9) Interpretation rapide

- `sigma_hat` est l'estimation de volatilite realisee latente extraite du SSM Heston via UKF.
- Le spread `sigma_iv - sigma_hat` sert de signal de timing.
- La strategie dynamique module l'exposition carry en fonction du z-score du spread.

Pistes d'amelioration pour le rendu final:
1. Tester plusieurs maturites cibles (20j, 30j, 45j) pour `sigma_iv`.
2. Faire une sensibilite sur `ROLL_WINDOW`, `RECALIB_FREQ`, `TC_BPS`.
3. Remplacer le `carry_proxy` par un PnL options plus proche d'une implementation (ex: short straddle ATM delta-hedge).
